# 1 · Sources — one uniform wrapper over every input

Every Digital-Earth plot reads from a **`Source`**: a small object holding the data grid `z`, the coordinate axes `x`/`y`, the `crs`, and free-form `metadata`. `digitalearth.sources.get_source` builds one from *any* supported input — a pyramids `Dataset` (raster), `NetCDF`, `DatasetCollection`, `FeatureCollection`, or a raw NumPy array — so the rest of the library never has to care where the data came from.

**pyramids is the only GIS dependency** (no xarray/rasterio): coordinates, CRS and nodata all come from the pyramids object.

This notebook builds a `Source` from each input type and inspects what comes out. The sample data is a **real elevation DEM of the Lisbon region** (421×547 cells, EPSG:4326).

**Covered here:** raster `Dataset` → `Source`, NaN-masking of nodata, coordinate vectors, a raw NumPy array, and one member of a `DatasetCollection`.

**Setup.** Enable inline plots and locate the bundled DEM by walking up to the repo root, so the notebook runs both from `docs/examples/` (mkdocs) and from the repository root.

In [1]:
%matplotlib inline
from pathlib import Path

# Resolve the repo root so the bundled sample DEM is found whether this runs from
# docs/examples/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DEM = str(ROOT / "examples" / "data" / "LisbonElevation.tif")
print("DEM:", DEM)

DEM: C:\gdrive\algorithms\serapeum-github-org\repos\visualization\Digital-Earth\examples\data\LisbonElevation.tif


## From a pyramids `Dataset` (the DEM raster)

Read the DEM as a pyramids `Dataset` and wrap it with `get_source`. The `Source` exposes the data grid (`z`), the 1-D coordinate axes (`x`/`y`), the CRS, and a `metadata` dict — note `kind=raster`.

In [2]:
from pyramids.dataset import Dataset
from digitalearth.sources import get_source

ds = Dataset.read_file(DEM)
src = get_source(ds)
print('kind     :', src.metadata('kind'))
print('z shape  :', src.z.values.shape)
print('x / y len:', src.x.values.shape, src.y.values.shape)
print('crs      :', src.crs)

kind     : raster
z shape  : (421, 547)
x / y len: (547,) (421,)
crs      : 4326


The data dimension is NaN-masked at the nodata value; `x`/`y` are 1-D cell-centre coordinates:

Inspect the wrapped values: nodata cells have been replaced with `NaN`, and `x`/`y` are the cell-centre longitude/latitude vectors (lengths = columns/rows).

In [3]:
import numpy as np
z = src.z.values
print('elevation range (m): %.1f .. %.1f' % (np.nanmin(z), np.nanmax(z)))
print('lon range:', round(float(src.x.values.min()), 3), '..', round(float(src.x.values.max()), 3))
print('lat range:', round(float(src.y.values.min()), 3), '..', round(float(src.y.values.max()), 3))

elevation range (m): -6.8 .. 217.9
lon range: -9.238 .. -9.086
lat range: 38.68 .. 38.797


## From a raw numpy array

No CRS; pixel-index axes unless you pass `x`/`y`.

The same wrapper accepts a **raw NumPy array** — useful for quick, non-georeferenced data. With no CRS and no coordinates given, the axes default to pixel indices and `crs` is `None`.

In [4]:
arr = np.arange(12.0).reshape(3, 4)
s = get_source(arr)
print('x:', s.x.values.tolist(), ' crs:', s.crs)

x: [0.0, 1.0, 2.0, 3.0]  crs: None


## From a `DatasetCollection` (one member)

A `DatasetCollection` (a stack of rasters) yields a `Source` for one member; the metadata records which member (`member`) and how many there are (`n_members`) so callers can iterate the rest.

In [5]:
from pyramids.dataset.collection import DatasetCollection

dc = DatasetCollection.from_files([DEM, DEM])
sc = get_source(dc)
print('member / n_members:', sc.metadata('member'), '/', sc.metadata('n_members'))

member / n_members: 0 / 2
